# رادار ۵.۴ — واکشی داده

**هر جلسه: سلول ۱ و ۲ را بزن. بعد ۳ (اسکن) یا ۴ (چرخش) یا ۵ (تحلیل عمیق).**

کولب روی سرور آمریکاست، پس بایننس و بای‌بیت مسدودند. `okx,gate` پوشش کامل می‌دهد.


## ۱ — نصب

In [ ]:
!pip install -q requests pandas numpy
print('✅ آماده')

## ۲ — دانلود تازه + کلید کوین‌گکو

این سلول **اول فایل‌های قدیمی را پاک می‌کند**، بعد نسخه تازه را می‌گیرد.

کلید کوین‌گکو اختیاری است — خالی بگذاری هم کار می‌کند، فقط کندتر.

In [ ]:
USER = 'AmirShalbaf'
REPO = 'radar'
COINGECKO_KEY = ''   # کلید رایگان Demo — اختیاری

import os, time
if COINGECKO_KEY.strip():
    os.environ['COINGECKO_API_KEY'] = COINGECKO_KEY.strip()

FILES = ['radar_fetch3.py', 'radar_scan.py', 'radar_rotate.py', 'radar_levels.py']
for f in FILES:
    !rm -f {f}

BASE  = f'https://raw.githubusercontent.com/{USER}/{REPO}/main'
STAMP = int(time.time())          # ضد کش گیت‌هاب
for f in FILES:
    !wget -q -O {f} "$BASE/{f}?v=$STAMP"

print('── فایل‌ها ──')
for f in FILES:
    ok = os.path.exists(f) and os.path.getsize(f) > 5000
    print(('✅ ' if ok else '❌ ') + f, os.path.getsize(f) if ok else '')

print('\n── بررسی اصلاح بلوغ میانگین (باید هر دو ۶۰۰ باشند) ──')
!grep -m1 -o 'order, [0-9]*' radar_scan.py
!grep -m1 -o 'order, [0-9]*' radar_rotate.py
!grep -m1 'VERSION = ' radar_fetch3.py


## ۳ — اسکن چند کوین  ← معمولاً از اینجا شروع کن

`PRESET`: `main` سه‌تایی | `watch` نه‌تایی | `all` سیزده‌تایی

In [ ]:
PRESET    = 'all'
WATCHLIST = ''            # اگر پر شود PRESET نادیده گرفته می‌شود.
                          # ZEC در preset نیست ولی پوزیشن باز است — مثال: 'ZEC,SUI,TAO'
VENUES    = 'okx,gate'    # بایننس و بای‌بیت از کولب مسدودند
TOP       = 3

sel = f'--watchlist {WATCHLIST}' if WATCHLIST.strip() else f'--preset {PRESET}'
!python radar_scan.py {sel} --venues {VENUES} --top {TOP} --stdout

## ۴ — شکارچی چرخش (کل بازار)

اسکن بالا فقط واچ‌لیست را می‌بیند. این یکی **کل بازار** را غربال می‌کند.

خروجی تازه دو چیز دارد که قبلاً نبود: جدول «هشدار بلوغ» و علامت ⚠ کنار نمادهای کم‌داده.


In [ ]:
VENUES  = 'okx,gate'
MIN_VOL = 3_000_000      # حداقل حجم روزانه دلاری
TOP     = 45             # چند نماد وارد تحلیل عمیق شود
DEEP    = 5              # چند نماد برتر گزارش کامل بگیرد
EXCLUDE = ''             # نمادهای حذفی، جدا با کاما

cmd = f'python radar_rotate.py --venues {VENUES} --min-vol {MIN_VOL} --top {TOP} --deep {DEEP} --stdout'
if EXCLUDE.strip():
    cmd += f' --exclude {EXCLUDE}'
!{cmd}


## ۵ — تحلیل عمیق یک کوین

نماد برنده اسکن را اینجا بگذار.

In [ ]:
SYMBOL      = 'XLM'
BALANCE     = 800
PROFILE     = 'trade'       # trade یا position
VENUES      = 'okx,gate'
MACRO_EVENT = '2026-07-29'  # لنگر سوم پروفایل حجم، یا خالی بگذار

cmd = f'python radar_fetch3.py {SYMBOL} --balance {BALANCE} --profile {PROFILE} --venues {VENUES} --stdout'
if MACRO_EVENT.strip(): cmd += f' --macro-event {MACRO_EVENT}'
!{cmd}

## ۶ — چند کوین پشت سر هم (اختیاری)

In [ ]:
for s in ['XLM','SUI','TAO']:
    print('\n' + '='*70 + f'\n{s}\n' + '='*70)
    !python radar_fetch3.py {s} --balance 800 --venues okx,gate --stdout

## ۷ — عیب‌یابی (فقط اگر لازم شد)

اگر مطمئن نیستی نسخه تازه است، این را بزن.

In [ ]:
!ls -la radar_*.py
!grep -m1 'VERSION = ' radar_fetch3.py
!head -3 radar_scan.py
import requests
for n,u in [('okx','https://www.okx.com/api/v5/public/time'),
            ('gate','https://api.gateio.ws/api/v4/spot/time'),
            ('binance','https://api.binance.com/api/v3/time'),
            ('bybit','https://api.bybit.com/v5/market/time')]:
    try:
        r = requests.get(u, timeout=10)
        print(f'{n:9} کد {r.status_code} ' + ('✅' if r.status_code==200 else '❌'))
    except Exception as e:
        print(f'{n:9} ❌ {type(e).__name__}')